In [42]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel, RationalQuadratic, DotProduct
import matplotlib.pyplot as plt

import sys
sys.path.append('../pyJive/')

from utils import proputils as pu
import main
from names import GlobNames as gn
from names import ParamNames as pn

import io
import contextlib

def update_geom_file(geom_path, y_coordinate, update_lines, output_path=None):
    output_path = output_path or geom_path  # Overwrite if no output path provided

    # Read the .geom file
    with open(geom_path, 'r') as file:
        lines = file.readlines()

    # Locate and modify the section with upper node coordinates
    for i in range(len(lines)):
        if i in update_lines:
            parts = lines[i].split()
            parts[-1] = str(y_coordinate)  # Modify the last value
            lines[i] = ' '.join(parts) + '\n'        
            
    # Write back the updated file
    with open(output_path, 'w') as file:
        file.writelines(lines)
        

def finite_element_solver(x):
    x = x[0]
    update_lines = [3, 5, 7 ,9, 11, 13, 15, 17, 19]
    update_geom_file('bridge.geom', x, update_lines, output_path=None)
    
    # Load properties
    props = pu.parse_file('bridge_frequency.pro')

    # Run FEM simulation
    with contextlib.redirect_stdout(io.StringIO()):
        globdat = main.jive(props)

    # Extract mass  
    mass_nodes = props['model']['mass']['nodeGroup']
    n_bottom_nodes = len(globdat[gn.NGROUPS][mass_nodes])
    point_mass = float(props['model']['mass']['mass'])
    weight = 0.5 * np.sum(globdat[pn.MATRIX2]) - n_bottom_nodes * point_mass
    
    # Extract natural frequencies
    frequencies = globdat[gn.EIGENFREQS][0:3] / (2 * np.pi)  # Hz

    return weight


def EI(x, gp, Y_samples):
    mu, sigma = gp.predict(x.reshape(1, -1), return_std=True)
    mu, sigma = mu[0], sigma[0]
    best = np.min(Y_samples)
    
    # Expected Improvement (EI)
    z = (best - mu) / (sigma + 1e-9)
    ei = (best - mu) * norm.cdf(z) + sigma * norm.pdf(z)
    
    return -ei


# def plot(gp, x_next, bounds):
#     X = np.linspace(bounds[0][0], bounds[0][1], 100)
#     mu, sigma = gp.predict(X, return_std=True)
    
#     plt.figure(figsize=(12, 6))
    
#     plt.plot(X, mu, 'b-', label="GP Mean")
#     plt.fill_between(
#         X.ravel(), mu - 1.96 * sigma, mu + 1.96 * sigma, alpha=0.2, label="95% Confidence Interval"
#     )   
    

    
def bayesian_optimization(n_iter=10):
    bounds = [[0.5, 5]]
    
    X_samples = np.random.uniform(0.1, 5.0, size=(5, 1))
    Y_samples = np.array([finite_element_solver(x) for x in X_samples])
    
    theta_0 = 2.25
    theta_1 = 4
    theta_2 = 2
    theta_3= 1
    rbf_kernel = ConstantKernel(constant_value=theta_0) * RBF(length_scale=1/np.sqrt(theta_1))
    constant_kernel = ConstantKernel(constant_value=theta_2)
    linear_kernel = DotProduct(sigma_0=theta_3)
    kernel = rbf_kernel + constant_kernel + linear_kernel
    
    gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-3, n_restarts_optimizer=10)
    
    for i in range(n_iter):
        gp.fit(X_samples, Y_samples)
        
        def acquisition(x):
            return EI(x, gp, Y_samples)
        
        x_next = minimize(acquisition, np.random.uniform(0.1, 5.0, 1), bounds=bounds).x
        y_next = finite_element_solver(x_next)
        
        X_samples_scaled = np.vstack((X_samples, x_next))
        Y_samples_scaled = np.append(Y_samples, y_next)
        
        # plot(gp, x_next, bounds)
       
    best_idx = np.argmin(Y_samples)
    return X_samples[best_idx], Y_samples[best_idx]
    
    
bayesian_optimization()

C:\Users\Gebruiker\anaconda3\envs\dsaie\lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\Gebruiker\anaconda3\envs\dsaie\lib\site-packages\sklearn\gaussian_process\_gpr.py:629: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
C:\Users\Gebruiker\anaconda3\envs\dsaie\lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a

(array([0.58513124]), 271.84784096318214)